**Non-Blocking Sleep with `asyncio`**

The main idea is: `asyncio.sleep()` pauses **only the current async task,** not the entire event loop.

In [1]:
import asyncio

async def wait_a_bit():
    print("Started")

    await asyncio.sleep(2)
    print("Finished")
    
await wait_a_bit()

Started
Finished


**Why is it called non-blocking?**

Compare these two:

🔴 Blocking

In [2]:
import time

def task():
    print("start")
    time.sleep(2)
    print("Finished")

In [3]:
task()

start
Finished


```
Task
 ↓
time.sleep(2)
 ↓
🔴 Event loop/thread is blocked
 ↓
Wait 2 seconds
 ↓
Continue
```

🟢 Non-blocking

In [4]:
import asyncio

async def task():
    print("start")
    await asyncio.sleep(2)
    print("Finished")

task()

<coroutine object task at 0x0000020F02C4B340>

In [5]:
await task()

start
Finished


```
Task
 ↓
await asyncio.sleep(2)
 ↓
🟢 Give control back to event loop
 ↓
Other async tasks can run
 ↓
2 seconds later...
 ↓
Continue task
```

**See the difference with two tasks**

In [7]:
import asyncio

async def task1():
    print("Task 1 started")
    await asyncio.sleep(3)
    print("Task 1 done")

async def task2():
    print("Task 2 started")
    await asyncio.sleep(1)
    print("Task 2 done")

async def main():
    await asyncio.gather(task1(), task2())

await main()

Task 1 started
Task 2 started
Task 2 done
Task 1 done


```
                EVENT LOOP
                    │
                    ▼
             Start Task 1
                    │
          await sleep(3)
                    │
             🟢 Give control
                    │
                    ▼
             Start Task 2
                    │
          await sleep(1)
                    │
             🟢 Give control
                    │
                    ▼
              1 second
                    │
                    ▼
             Task 2 done
                    │
                    │
                    ▼
              3 seconds
                    │
                    ▼
             Task 1 done
             ```